In [1]:
import torch
import torch.nn.functional as F
import json
from PIL import Image
from torchvision import transforms
from torchvision.models import (
    efficientnet_b0,
    EfficientNet_B0_Weights
)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

# Classes

with open(
    "classes.json",
    "r"
) as f:

    classes = json.load(f)
# Model

model = efficientnet_b0()

model.classifier[1] = torch.nn.Linear(
    model.classifier[1].in_features,
    len(classes)
)

model.load_state_dict(
    torch.load(
        "best_document_classifier.pth",
        map_location=DEVICE
    )
)

model.to(DEVICE)
model.eval()

# Transform

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

C:\Users\nitis\AppData\Local\Temp\ipykernel_18012\294053716.py:34: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(


In [2]:
def predict_image(image_path):

    img = Image.open(
        image_path
    ).convert("RGB")

    x = transform(img)

    x = x.unsqueeze(0)

    x = x.to(DEVICE)

    with torch.no_grad():

        output = model(x)

        probs = F.softmax(
            output,
            dim=1
        )

        confidence, pred = torch.max(
            probs,
            dim=1
        )

    confidence = (
        confidence.item() * 100
    )

    if confidence < 80:

        prediction = "unknown"

    else:

        prediction = classes[
            pred.item()
        ]

    print(
        f"Prediction: {prediction}"
    )

    print(
        f"Confidence: {confidence:.2f}%"
    )
    
    probs = probs[0]

    for i, p in enumerate(probs):

        print(
            f"{classes[i]:<20}"
            f"{p.item()*100:.2f}%"
        )

    return prediction, confidence

In [25]:
predict_image(
    r"C:\Users\nitis\Downloads\DQlPl0gUEAAdyQ9.jpg"
)

Prediction: driving_license
Confidence: 95.79%
aadhaar             0.14%
driving_license     95.79%
pan                 0.08%
passport            1.14%
voterId             2.85%


('driving_license', 95.79455852508545)